In [1]:
# # farpoint Algorithm on MNIST
import os
import random
import pickle
import math
import itertools
import time

try:
    from datetime import datetime
    import numpy as np
    from scipy.spatial import distance
    from scipy.spatial.distance import pdist, squareform
    from scipy.cluster.hierarchy import dendrogram, linkage
    from scipy.cluster.hierarchy import fcluster
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.cluster import KMeans
    from sklearn.metrics import confusion_matrix
    from sklearn.manifold import TSNE
    from sklearn import datasets, metrics
    from active_semi_clustering.semi_supervised.pairwise_constraints import COPKMeans
except:
    print("resolving system dependencies")
    os.system("pip install -r requirements.txt")
    from active_semi_clustering.semi_supervised.pairwise_constraints import COPKMeans
    import numpy as np
    import matplotlib as mpl
    import matplotlib.pyplot as plt
    from sklearn.cluster import KMeans
    from sklearn.metrics import confusion_matrix
    from scipy.spatial import distance
    from scipy.spatial.distance import pdist, squareform
    import seaborn as sns
    from sklearn.manifold import TSNE
    from scipy.cluster.hierarchy import dendrogram, linkage
    from scipy.cluster.hierarchy import fcluster
    from sklearn import datasets, metrics
    from datetime import datetime
print("imports successful!")

imports successful!


In [2]:
whos_computer = 'Ro'
# whos_computer = 'Noah'
# whos_computer = 'other'
# source = 'human'
source = 'comp'
DECIMATION_FACTOR = 1
MAX_NUM_QUERIES = 1000
NUM_CLASSES = 10
NUM_RUNS = 1# this is the number of runs per method stats object(num runs in pkl file)
NUM_TRIALS = 500 #this is the number of method stats objects to create (num pkl files)
# choose one or the other
# SPLIT_METHOD = "NQ" # Nearest_Queried
SPLIT_METHOD = "COPKM" #'COPK_Means'
# SPLIT_METHOD = "HA" #'Single_Link'

In [3]:
#np.random.seed(10) # DON'T USE OVER MULTIPLE COMPUTER RUNS
# %matplotlib qt
# %matplotlib inline

In [4]:

if whos_computer == "Ro":
    npzfile = np.load("/home/johnnybravo/data/mnist.npz")
    raw_data = npzfile['mnist_data']
    raw_labels = npzfile['mnist_labels']
elif whos_computer == 'Noah': #Noah's computer    
    npzfile = np.load("/home/noahkaplan/Desktop/mnist.npz")
    raw_data = npzfile['mnist_data']
    raw_labels = npzfile['mnist_labels']
else:
    response = datasets.fetch_openml("mnist_784")
    raw_data = response['data'].astype(float)
    raw_labels = response['target'].astype(int)

In [5]:
# generate skewed data
# NOTE: each class has half prob of next

# Generate skewed data -------------------------------------------------------------------
def skew_data(data, labels):
    # set class frequencies - each class has half prob of next
    skew_vec = np.empty((NUM_CLASSES,))
    for n in range(NUM_CLASSES):
        skew_vec[n] = 2**n
    skew_vec = skew_vec/np.amax(skew_vec)
#     print(skew_vec)


    shfl_cls = np.arange(NUM_CLASSES) # randomly shuffle class numbers 
    np.random.shuffle(shfl_cls)
#     print(shfl_cls)
    skewed_label_list = []
    skewed_data = np.empty((0,data.shape[1]))
    class_probs = np.empty(NUM_CLASSES)
    # use all available samples of most represented class (i.e. at skew_vec[9])
    most_fr_class_ind = shfl_cls[9]
    n_samples_most_frq_class = len(np.where(labels == most_fr_class_ind)[0])
    # this number corresponds to the 100% class - all other %'s in skew_vec are relative to that
    for c in range(NUM_CLASSES):
        shfld_c = shfl_cls[c]
        this_c_Mi= np.where(labels == shfld_c)[0]
        n_samples_total = len(this_c_Mi)
        class_probs[shfld_c] = skew_vec[c]
        n_samples_skewed = math.floor(n_samples_most_frq_class*skew_vec[c])
    #     print(n_samples_skewed)
        tmp = [shfld_c]*n_samples_skewed
        skewed_label_list = skewed_label_list + tmp
        sk_inds_c = np.random.choice(n_samples_total,n_samples_skewed) # these are k random indices out of the total number 
        # of samples with this class label, where k is n_samples_skewed
        sk_inds_c_Mi = this_c_Mi[sk_inds_c]
        data_c = data[sk_inds_c_Mi,:]
        skewed_data = np.concatenate((skewed_data,data_c)) # I know - it's not efficient =^(
    skewed_labels = np.array(skewed_label_list)
    
    return skewed_data,skewed_labels,skew_vec,class_probs

skewed_data,skewed_labels,skew_vec,class_probs = skew_data(raw_data, raw_labels)
# print(skewed_data.shape)
# print(skewed_labels.shape)
# print(skewed_labels[0:50])
print(class_probs*100)

print(skew_vec/sum(skew_vec)*100)


# NOTE: "probs" here in both skew_vec and class_probs are actually 2*real prob. and real prob is really
# off by a small amount from adding up to 1 because we cut off series after 10 terms

[100.          0.390625    0.78125    12.5         3.125       6.25
  50.         25.          1.5625      0.1953125]
[ 0.09775171  0.19550342  0.39100684  0.78201369  1.56402737  3.12805474
  6.25610948 12.51221896 25.02443793 50.04887586]


In [6]:
# decimate and rename to make data more manageable
data = skewed_data[::DECIMATION_FACTOR,:]
labels = skewed_labels[::DECIMATION_FACTOR]
assert len(np.unique(labels)) == NUM_CLASSES, "Too much decimation - don't have all classes" 
print(data.shape)
print(labels.shape)

(13786, 784)
(13786,)


In [7]:
def group_by_label(data, labels):
    #params:
    #   data - np.array shape (num_elements, 2) 2 features per element
    #   labels - np.array  shape (num_elements) with labels matching to data
    #       where label is -1, point is not included in group
    # returns:
    #     grouped_data - dictionary with:
    #           each key: a unique label
    #           corr. values: all data pts that have that label
    #         the data is mapped to have an np.array shape = (num_grouped_elements, 2)
        grouped_data = {}
        i = 0
        #iterate through data and store each data value to dictionary under label
        while i < data.shape[0]:
            if labels[i] != -1:
                if labels[i] in grouped_data.keys():
                    grouped_data[labels[i]].append(data[i].reshape((1,2)))
                else:
                    grouped_data[labels[i]] = [data[i].reshape((1,2))]
            i += 1
        #iterate through dictionary and concatenate list of elements into one np.array
        for k in grouped_data.keys():
            grouped_data[k] = np.concatenate(grouped_data[k], axis=0)
        return grouped_data

In [8]:
#It looks bad, but it's probably the best way to go about it
MNIST_colors={0:(0,0,.8),
                  1:(0,0.5,1),
                  2:(0.2,0.2,0),
                  3:(0.2,1,0.2),
                  4:(1,0,0),
                  5:(0.4,0,.8),
                  6:(0.9,0.4,.8),
                  7:(0.9,0.9,.1),
                  8:(1,.4,0),
                  9:(0.3,0.6,0.2)
                 }

#This may be helpful for fixing the missing legend problem
# MNIST_color_patches = {}
# for k in MNIST_colors.keys():
#     MNIST_color_patches[k]=mpl.patches.Patch(color=MNIST_colors[k])
#     print(MNIST_colors[k])
#     print(MNIST_color_patches[k])

In [9]:
def plot_by_label(data, labels, color_key, title="Data Plotted By Labels"):
    #scatter plots data color coordinated by label
    #parameters:
    #    data - np.array of features (no titles)
    #    labels - np.array with labels to data (aligned by index)
    
    #tSNE if data is not yet 2d
    if data.shape[1] != 2:
        data = TSNE(n_components=2).fit_transform(data)
    #configure plot
    plt.figure(figsize = (14,7))
    plt.grid(True)
    plt.title(title)
    #grouped_data holds dictionary keys mapped to np.array (num_grouped_elements, 2 features)
    grouped_data = group_by_label(data, labels)
    #plot each group seperately
    for k in grouped_data.keys():
        plt.plot(grouped_data[k][:,0], grouped_data[k][:,1],'.', c=color_key[k], label=k)
#     plt.legend(grouped_data.keys())
    plt.legend()
    plt.show()
    
#TEST
# plot_by_label(data,labels, MNIST_colors)

In [10]:
def find_1pt_dists(pt,other_pts):
    # parameters:
    #     pt - np array shape num_features,
    #     other_pts - np array num_samples x num_features
    # returns:
    #     dists - np array num_samples x 1 - distances between pt and each of the other_pts
    dists = np.zeros((len(other_pts),1))
    for i in range(len(other_pts)):
        dists[i,0] = distance.euclidean(pt, other_pts[i,:])
    return dists


In [11]:
def display_MNIST_data_pt(Mi):
    im_data = data[Mi,:].reshape((28,28))
    plt.imshow(im_data)
    plt.show()
    
# display_MNIST_data_pt(0)

In [12]:
def query_pt(Mi,source = 'comp'):
    if source == 'comp': # get from already known labels
        label = labels[Mi]
    elif source == 'human': # present to human for label
        display_MNIST_data_pt(Mi)
        label = input("please enter label: (0-9)")
        assert label == labels[Mi], "computer label is different from human, Mi#: "+str(Mi)
        plt.close()
    return label


In [13]:
# Nearest Center Assignment (NCA)
def NCA(cl_centers,all_pts):
    # parameters:
    #    cl_centers - 2-D array - each row is a cluster center 
    #      (num rows = num clusters, num cols = num features)
    #    all_pts - 2-D array - each row is a sample
    #      (num rows = num samples, num cols = num features)
    # returns:
    #    pt_labels - 1-D array of labels 
    #      (num elements = num pts)    
    n_clusters = len(cl_centers)
    n_pts = len(all_pts)
    # create distance array with pts in rows, cl centers in cols
    cl_dists = -1*np.ones((n_pts,n_clusters))
    for n in range(n_clusters):
        cl_dists[:,n] = np.squeeze(find_1pt_dists(cl_centers[n,:],all_pts))
        
    pt_labels = np.argmin(cl_dists,axis = 1)
    return pt_labels

In [14]:
def single_link(X,num_l_pts):
    #all_pts[0] = f_pt
    #all_pts[1:num_seeds] = l_pts
    #all_pts[num_seeds:] = o_pts
    def single_link_cls_dist(cl1_cl_num,cl2_cl_num,X,cl_assignments):
        #Find the single linkage dist between two clusters
        
        cl1_inds = np.where(cl_assignments == cl1_cl_num)
        cl1_pts = X[cl1_inds]
        cl2_inds = np.where(cl_assignments == cl2_cl_num)
        cl2_pts = X[cl2_inds]
        dist_matrix = distance.cdist(cl1_pts, cl2_pts, 'euclidean')
        single_link_dist = np.min(dist_matrix)
        return(single_link_dist)
    
    def Z_search(search_list,Z):
        #params: list search_list - elements to look for
            #np.array Z - linkage matrix
        #returns: np.array() - column with 1s where linkages cluster contains any elements of search list
        search_copy = search_list.copy() # don't add to original label or f_pt lists
        num_pts = Z.shape[0] + 1
        incl_col = np.zeros((len(Z),1))
        for r in range(len(Z)):
            if int(Z[r,0]) in search_copy or int(Z[r,1]) in search_copy:
                incl_col[r] = 1
                search_copy.append(r+num_pts)
        return incl_col
    
    f_pt_index = [0]
    l_pt_inds = list(range(1,num_l_pts+1))
    Z = linkage(X, 'single')
    
    #add columns with 1 next to elements in f_pt_index
    Z_aug = np.concatenate((Z,Z_search(f_pt_index,Z)),axis = 1)
    #add columns with 1 next to elements in l_pts
    Z_aug = np.concatenate((Z_aug,Z_search(l_pt_inds,Z)),axis = 1)
    
    #find threshold by searching new columns
    z_f_pt_inds = np.where(Z_aug[:,4] == 1.0)
    z_l_pts_inds = np.where(Z_aug[:,5] == 1.0)
    threshold_ind = np.intersect1d(z_f_pt_inds, z_l_pts_inds)[0]
    thresh = Z_aug[threshold_ind-1,2]
    
    #get cluster nums
    cl_assignments = fcluster(Z, t=thresh, criterion='distance')
    f_pt_cluster_num = cl_assignments[f_pt_index]
    l_pt_cluster_nums = np.unique(cl_assignments[l_pt_inds])    
    
    #Combine l_pts clusters to have same cluster number
    for n in l_pt_cluster_nums:
        this_cl_inds = np.where(cl_assignments == n)
        cl_assignments[this_cl_inds] = l_pt_cluster_nums[0]
    l_pt_cluster_num = l_pt_cluster_nums[0]

        # find orphan cluster labels
    orphan_cl_nums = np.setdiff1d(cl_assignments,f_pt_cluster_num)
    orphan_cl_nums = np.setdiff1d(orphan_cl_nums,l_pt_cluster_nums)

    # assign orphan to closest (by single-linkage dist) cluster, so either f_pt cluster or labels cluster
    for orphan_cl_num in orphan_cl_nums:
        # find distance from orphan to far point cluster
        orphan_f_pt_dist = single_link_cls_dist(orphan_cl_num,f_pt_cluster_num,X,cl_assignments)
        # find distance from orphan to labels cluster
        orphan_labels_dist = single_link_cls_dist(orphan_cl_num,l_pt_cluster_num,X,cl_assignments)
        orphan_cl_inds = np.where(cl_assignments == orphan_cl_num)

        if orphan_f_pt_dist < orphan_labels_dist: # assign orphan to f_pt
            cl_assignments[orphan_cl_inds] = f_pt_cluster_num
        else: # assign orphan to labeled pts
            cl_assignments[orphan_cl_inds] = l_pt_cluster_num
    
    return cl_assignments

In [15]:
class Cluster:
    # NOTE: clusters have 2 mutually exclusive pools of points:
    #  l_pts - labeled points, known for sure
    #  o_pts - pts in this cluster, presumed to have this label due to clustering
    #       - if there are no o_pts, try to return -1 when asked for one
    #  f_pt_Mi - index of one of the o_pts, farthest from the l_pts
    #  f_pt_dist - dist to f_pt
    # NOTE: -1 for f_pt_Mi or o_pt_Mi indicates invalid index (possibly because we're down to cluster of 
    # size 1!)
    def __init__(self,label,l_pts_Mi,o_pts_Mi):
        self.label = label
        self.l_pts_Mi = l_pts_Mi # a list of M_inds to pts known to have this label
        self.o_pts_Mi = o_pts_Mi # a list of M_inds to pts labelled by clustering
        self.update_far_point()# this creates self.f_pt_Mi and self.f_pt_dist
        
    def summary(self):
        print('label: ',self.label,'l pts: ',self.l_pts_Mi,' f_pt index: ',self.f_pt_Mi,
              'number of o_pts: ',len(self.o_pts_Mi))
        
    def update_far_point(self):
        # parameters:
        #     l_pts_Mi - indices of pts in data of 'labeled' pts
        #     o_pts_Mi - indices of pts in data of 'other' pts
        # returns:
        #     various kinds of information about the "far point" - i.e. the point out of all the "other"
        # pts pool that is furthest from the "labeled" pts, in a max min sense
        # (i.e.the minimum distance from the far point to any of the labeled points is >= the minimum distance 
        # from any of the other points to the labeled points)
        #     f_pt - coord values of the far pts
        #     f_pt_M_ind - index of far point in data
        #     f_pt_dist - max min dist of far pt from labeled pts
        #    NOTE: if there are no o_pts, this cluster is "done" and this returns empty list for f_pt, an
        #    index of -1, and 0 dist
        # algorithm:
        #    construct matrix of dists ("tmp_dist_matrix") where rows are l_pts, cols are o_pts
        #    find min of each column - that's the distance from a given o_pt to the nearest l_pt, so that's 
        #      the "worst" case for that o_pt
        #    store all those min's in a row vector
        #    then find the max of that vector - that goes with the o_pt that is the far point

        if not self.o_pts_Mi:  # this means there are no unknown/unlabelled points (ie o pts) in this cluster, so
            # there will be no far pt
            self.f_pt_Mi = -1
            self.f_pt_dist = 0
        else:
            self.l_pts = data[self.l_pts_Mi,:]
            # if only 1 index array shape will be wrong so check and reshape if necessary
            if len(self.l_pts.shape) == 1:
                self.l_pts = np.reshape(self.l_pts,(1,-1))
            self.o_pts = data[self.o_pts_Mi,:]
            if len(self.o_pts.shape) == 1:
                self.o_pts = np.reshape(self.o_pts,(1,-1))
            tmp_dist_matrix = -1*np.ones((len(self.l_pts),len(self.o_pts))) # l pts rows, o pts cols
            for r in range(len(self.l_pts)):
                tmp_dist_matrix[r,:] = find_1pt_dists(self.l_pts[r,:],self.o_pts).transpose()
            # find min dist in each column
            min_dists_row = np.amin(tmp_dist_matrix,axis = 0)
            # then find index of max in resulting row
            f_pt_local_ind = np.argmax(min_dists_row)
            self.f_pt_Mi = self.o_pts_Mi[f_pt_local_ind]
            self.f_pt_dist = np.max(min_dists_row)
    
    def incorporate_far_point(self):
        # remove f_pt from o_pts
        if self.f_pt_Mi != -1:
            self.o_pts_Mi.remove(self.f_pt_Mi)
        # add f_pt to l_pts
        self.l_pts_Mi.append(self.f_pt_Mi)
        # calc new f_pt, f_pt_dist
        self.update_far_point()
        
    
    def split_COPK(self,f_pt_label):
        # NOTE: this corresponds to Farpoint-COPKM
#         print("Entering split_COPK")        
#         self.summary()
        #  reconstruct an array w/ f_pt_Mi, followed by l_pts_Mi, followed by o_pts_Mi
        seed_inds_Mi = [self.f_pt_Mi] + self.l_pts_Mi
        num_seeds = len(seed_inds_Mi)
        
        # need o_pts WITHOUT f_pt        
        o_pts_Mi_wo_fp = self.o_pts_Mi.copy()
        o_pts_Mi_wo_fp.remove(self.f_pt_Mi)
        # then use o_pts_Mi_wo_fp for o_pts.
        all_pts_Mi = np.array(seed_inds_Mi + o_pts_Mi_wo_fp)
        # get corresponding actual data
        all_pts = data[all_pts_Mi,:]

        # form constraint list based on those local indices of f_pt and l_pts
        cant_links = list(itertools.combinations(range(num_seeds), 2))

        # do COPK
        clusterer = COPKMeans(n_clusters=num_seeds)
#             # make initial seed list
#             seed_pts = []
#             for n in range(num_seeds):
#                 seed_pts.append(all_pts[n,:])
        seed_pts = all_pts[:num_seeds, :]
        clusterer.fit(all_pts,y = seed_pts, ml=[], cl=cant_links)
        # print(clusterer.labels_)

        o_pt_clusterer_labels = np.array(clusterer.labels_[num_seeds:])
        f_pt_local_label = clusterer.labels_[0]
        fp_cl_local_o_pt_inds = np.where(o_pt_clusterer_labels == f_pt_local_label)[0]
        non_fp_cl_local_o_pt_inds = np.where(o_pt_clusterer_labels != f_pt_local_label)[0]
        o_pts_Mi_arr = np.array(o_pts_Mi_wo_fp)
        fp_cl_o_pt_Mi = list(o_pts_Mi_arr[fp_cl_local_o_pt_inds])
        non_fp_cl_o_pt_Mi = list(o_pts_Mi_arr[non_fp_cl_local_o_pt_inds])
        # Now have 4 mut. excl. populations  
        #  orig cl l pts (not f_pt_local_label)
        #  orig cl o pts (not f_pt_local_label)
        #  f_pt (f_pt_local_label) - will become starting l pt of new cluster
        #  new cl o pts (f_pt_local_label)
        # form resulting clusters
        fp_cluster = Cluster(f_pt_label,[self.f_pt_Mi],fp_cl_o_pt_Mi)
        non_fp_cluster = Cluster(self.label,self.l_pts_Mi,non_fp_cl_o_pt_Mi)        
        
#         fp_cluster.summary()
#         non_fp_cluster.summary()
        
        return fp_cluster, non_fp_cluster  

    
    def split_nearest(self,f_pt_label):
        # NOTE: this corresponds to Farpoint-NQ
        # returns 2 clusters
        
        # recluster (this cluster only) w/ all l_pts _and_ f_pt as seeds ---------------------------
        seed_inds = [self.f_pt_Mi] +self.l_pts_Mi
        cl_centers = data[seed_inds,:]      
        n_clusters = len(cl_centers)
        all_pts = np.concatenate((cl_centers,data[self.o_pts_Mi,:]))
        all_pts_Mi = np.array([self.f_pt_Mi] + self.l_pts_Mi + self.o_pts_Mi)
        
        pt_labels = NCA(cl_centers,all_pts)
        
        # break clusters into new f pt cluster, and everything else (stays w/ orig. cl l pts)-------
        local_f_pt_label = 0 #f_pt was at start of list
        # find the rest of the points - from o_pts only! - that ended up in f_pts local cluster
        # NOTE: need [0] on the end because np.where returns tuple size 1(?)
        local_fp_cl_pt_inds = np.where(pt_labels == local_f_pt_label)[0] # local_new_cl_pt_inds is indexed into all_pts_Mi
        
        # Now have 4 mut. excl. populations, local labeled w/ 0 (associated w/ f_pt and new cluster), and 
        # local label not 0 (associated w/ orig. cl):
        #  orig cl l pts (not local label 0)
        #  orig cl o pts (not local label 0)
        #  f_pt (local label 0) - will become starting l pt of new cluster
        #  new cl o pts (local label 0)
        fp_cl_Mi = all_pts_Mi[local_fp_cl_pt_inds]
        fp_cluster_l_pts_Mi = [fp_cl_Mi[0]]
        fp_cluster_o_pts_Mi = list(fp_cl_Mi[1:])
        
        # form fp cluster ---------------------------------------------------------------
        # NOTE: updating the far point of new cluster and removing new f_pt from o_pts occurs in __init()__
        fp_cluster = Cluster(f_pt_label,fp_cluster_l_pts_Mi,fp_cluster_o_pts_Mi)
        
        # form update current cluster ----------------------------------------------------------------------
        # remove o_pts of new cluster from current cluster
        # NOTE: nice way off net - 1st attempt using 'for loop' w/ remove's not great...
        self.o_pts_Mi= list(set(self.o_pts_Mi).difference(set(fp_cl_Mi)))
        
        non_fp_cluster = Cluster(self.label,self.l_pts_Mi,self.o_pts_Mi)
        
        return fp_cluster, non_fp_cluster  

    def split_single_link(self,f_pt_label):
        # NOTE: this corresponds to Farpoint-HA
#         self.summary()

        #  reconstruct an array w/ f_pt_Mi, followed by l_pts_Mi, followed by o_pts_Mi
        num_l_pts_Mi = len(self.l_pts_Mi)
      
        
#         print(self.l_pts_Mi)
#         print(self.o_pts_Mi)
#         l_set = set(self.l_pts_Mi)
#         o_set = set(self.o_pts_Mi)
#         print(l_set.intersection(o_set))
#         print('!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!11!')

        # need o_pts WITHOUT f_pt        
        o_pts_Mi_wo_fp = self.o_pts_Mi.copy()
        o_pts_Mi_wo_fp.remove(self.f_pt_Mi)
        # then use o_pts_Mi_wo_fp for o_pts.
        all_pts_Mi = np.array([self.f_pt_Mi] + self.l_pts_Mi + o_pts_Mi_wo_fp)
        
        # get corresponding actual data
        all_pts = data[all_pts_Mi,:]
        
        # so now have all_pts arranged so that
        # all_pts[0] == f_pt
        # all_pts[1:num_l_pts_Mi+1] == l_pts
        # all_pts[num_l_pts_Mi+1:] == o_pts
        clusterer_labels = single_link(all_pts, num_l_pts_Mi)
        
        o_pt_clusterer_labels = np.array(clusterer_labels[num_l_pts_Mi+1:])
        f_pt_local_label = clusterer_labels[0]
        fp_cl_local_o_pt_inds = np.where(o_pt_clusterer_labels == f_pt_local_label)[0]
        non_fp_cl_local_o_pt_inds = np.where(o_pt_clusterer_labels != f_pt_local_label)[0]
        o_pts_Mi_arr = np.array(o_pts_Mi_wo_fp) #=all_pts[num_l_pts_Mi+1:]
        fp_cl_o_pt_Mi = list(o_pts_Mi_arr[fp_cl_local_o_pt_inds])
        non_fp_cl_o_pt_Mi = list(o_pts_Mi_arr[non_fp_cl_local_o_pt_inds])
        # Now have 4 mut. excl. populations  
        #  orig cl l pts (not f_pt_local_label)
        #  orig cl o pts (not f_pt_local_label)
        #  f_pt (f_pt_local_label) - will become starting l pt of new cluster
        #  new cl o pts (f_pt_local_label)
        # form resulting clusters
        fp_cluster = Cluster(f_pt_label,[self.f_pt_Mi],fp_cl_o_pt_Mi)
        non_fp_cluster = Cluster(self.label,self.l_pts_Mi,non_fp_cl_o_pt_Mi)        
        
#         fp_cluster.summary()
#         non_fp_cluster.summary()
        
        return fp_cluster, non_fp_cluster


            
    def split(self,f_pt_label,split_method):
        if split_method == 'NQ':
            return self.split_nearest(f_pt_label)
        elif split_method == 'COPKM':
            return self.split_COPK(f_pt_label)
        elif split_method == 'HA':
            return self.split_single_link(f_pt_label)

        
    def query_cluster(self):
        # query this cluster and return the f_pt label
        label = query_pt(self.f_pt_Mi,source = source)
        return label
    
    def display(self, data, color):
    #Assume that data has been TSNED
        o_pts=data[self.o_pts_Mi]
        f_pt=data[self.f_pt_Mi]
        l_pts=data[self.l_pts_Mi]
        plt.plot(o_pts[:,0], o_pts[:,1], '.', label=self.label, c=color, Alpha=0.6)
        plt.plot(f_pt[0], f_pt[1], '+', label=self.label, ms=15, c=color)
        plt.plot(l_pts[:,0], l_pts[:,1], '^', label=self.label, ms=10, c=color)


        
        

In [16]:
class Partition:
    # this holds all the clusters in a partition (so pairwise disjoint and includes all elements)
    def __init__(self,data,labels,split_method, method_stats=None):
        # make "first_cluster" by...
            # select a random point (rp)
#         self.data = data
#         self.labels = labels
        self.split_method = split_method
        
        rp_ind = random.randint(0,len(data)-1)
        self.rp_ind = rp_ind
            # create a cluster that holds all the pts as o-pts except rp, which is the l-pt
            # just use fake label for now - we'll replace it in a minute
        fake_label = 0
        o_pt_inds = list(range(0,rp_ind))+list(range(rp_ind+1,len(data)))
        first_cluster = Cluster(fake_label,[rp_ind],o_pt_inds) 
        self.first_cluster = first_cluster
            # query (get label for f_pt)
        label = first_cluster.query_cluster()
            # return rp to o_pts
        first_cluster.o_pts_Mi.append(rp_ind)
        first_cluster.l_pts_Mi.remove(rp_ind)
            # make f_pt _the_ labeled pt for the start cluster
            # re-update f_pt, f_pt_dist in this cluster
        first_cluster.incorporate_far_point()
        first_cluster.label = label
                
        # add this cluster to list of clusters in partition
        self.cl_list = [first_cluster]
        
        # this keeps track of the number of queries required to find a given label
        # keys are labels - values are numbers of queries
        self.class_v_query_dict = {}
        self.class_v_query_dict[label] = 1
                
        #stat collection
        self.stats = method_stats
        if self.stats:
            self.stats.update_on_query(0, acc_vec=self.get_acc_vec(labels,data,self.stats.num_classes),
                                          total_acc = self.get_acc(labels,data))
            
    
    def process_query(self,query_cl_ind,f_pt_label, query_num, num_classes=NUM_CLASSES):
        # use label to either incorporate or split
        if self.cl_list[query_cl_ind].label == f_pt_label: # incorporate
            self.cl_list[query_cl_ind].incorporate_far_point()
            # update acc list with last value, since nothing changed
            if self.stats:
                self.stats.update_on_query(query_num-1)
        else: # split
            fp_cluster, non_fp_cluster = self.cl_list[query_cl_ind].split(f_pt_label,self.split_method)
            # insert the 2 new clusters into the cluster list
            self.cl_list.append(fp_cluster)
            self.cl_list.append(non_fp_cluster)
            # delete the split cluster
            del(self.cl_list[query_cl_ind])
            # update acc list and have to recalculate
            if self.stats:
                self.stats.update_on_query(query_num-1, acc_vec=self.get_acc_vec(labels,data,num_classes),
                                          total_acc = self.get_acc(labels,data))
                
    def select_query_cluster_ind(self):
        dists = []
        for x in self.cl_list:
            dists.append(x.f_pt_dist)
        max_ind = np.array(dists).argmax()
        return max_ind

        
    def predict_labels(self, data):
    # goes through clusters and
    # returns:
    #    labels - 1-d array of predicted labels
        l_pts_labels = -1*np.ones((data.shape[0]), dtype=int)
        o_pts_labels = -1*np.ones((data.shape[0]), dtype=int)
        f_pts_labels = -1*np.ones((data.shape[0]), dtype=int)
        for cl in self.cl_list:
            this_label = cl.label
            l_pts_labels[cl.l_pts_Mi] = this_label
            o_pts_labels[cl.o_pts_Mi] = this_label
            f_pts_labels[cl.f_pt_Mi] = this_label
        return l_pts_labels, o_pts_labels, f_pts_labels
    
    def predict_labels_all_pts(self, data):
        #This is a little ugly
        #Concatenate three arrays and then join them by the max value (the -1 is ignored)
        pred_ls_1d = self.predict_labels(data)
        pred_ls = [arr.reshape(1, len(arr)) for arr in pred_ls_1d] #reshape to 2d arrs
        all_pts_2d = np.concatenate(pred_ls, axis=0)
        all_pts = np.amax(all_pts_2d, axis=0)
        return all_pts
    
    def display(self, data, color_key, title="Partition By Clusters"):
        plt.figure(figsize = (14,7))
        plt.grid(True)
        plt.title(title)
        for cl in self.cl_list:
            label=cl.label
            color = color_key[label]
            cl.display(data, color)
        plt.show()
            
    def display_by_label(self, data, color_key, title="Partition By Labels"):
        plot_by_label(data, self.predict_labels_all_pts(data), color_key, title)        
    
    def get_acc(self, labels,data):
        #Param: np.array labels - the skewed labels
        #Return: float accuracy - num_correct_preds/total_num_pts
        partition_labels = self.predict_labels_all_pts(data)
        acc = np.sum(labels == partition_labels) / len(labels)
        return acc
    

    def get_acc_vec(self, labels, data, num_classes=NUM_CLASSES):
        #params:
            #array labels: the skewed labels
            #array data: the skewed data
            #int class_num: the class id
            #int num_classes: the total number of classes in this data set
        #return:
            #array vector shape (num_classes,) holds the accuracies of each class num at the corresponding index,
              #and holds accuracy on the full data at the last index. If class has not been found, returns zero automatically
        acc_vec = np.zeros(num_classes)
        partition_labels = self.predict_labels_all_pts(data)
        for class_num in self.class_v_query_dict.keys():
            relevant_Mi = np.where(labels == class_num)
            acc_vec[class_num] = np.sum(labels[relevant_Mi] == partition_labels[relevant_Mi]) / len(labels[relevant_Mi])
        return acc_vec
        
    def run(self, num_classes = NUM_CLASSES,max_num_queries = MAX_NUM_QUERIES):
        if self.stats:
            start = time.time()
            time_to_find_all_classes = np.nan
        num_queries = 2 # 2 because initializing did 1st query already        
        while num_queries <= MAX_NUM_QUERIES:
            # query next cluster and process result
            q_cl_ind = self.select_query_cluster_ind()
            f_pt_label = self.cl_list[q_cl_ind].query_cluster()
            self.process_query(q_cl_ind, f_pt_label, num_queries, num_classes)
            if f_pt_label not in self.class_v_query_dict: # new label we haven't seen before
                self.class_v_query_dict[f_pt_label] = num_queries
                if len(self.class_v_query_dict) == num_classes: # if no -1's left then exit
                    if self.stats:
                        time_classes_found = time.time()
                        time_to_find_all_classes = time_classes_found - start
                    print(self.class_v_query_dict)
                    print("all 10 classes found at " + str(num_queries) + " queries")
            if num_queries % 10 == 0:
                print(num_queries)
            num_queries += 1
        #exit code
        if self.stats:
            end = time.time()
            time_to_complete_all_queries = end-start
            self.stats.update_on_end_of_run(self.class_v_query_dict, time_to_complete_all_queries, time_to_find_all_classes)
            
        
    

In [17]:
class CPR:
    """
    The base of CPR (class, probabilities, runs) is a 3D np.array (CPR_arr) with classes on i axis, 
      probabilities on j axis and run number on the k axis
    Each iXj layer corresponds to a single run. The cell (i,j) in the array is 0 unless class i was at probability j
      for this run. In this case, cell (i,j) contains an integer, the number of queries it took to find that class
      in this run
    ATTRIBUTES:
        array CPR_arr: shape (NUM_CLASSES, NUM_CLASSES, NUM_RUNS) the backbone of CPR.
        array num_trials_CPR_arr: (NUM_CLASSES, NUM_CLASSES) keeps track of the number of nonzero elements in each
          pillar of CPR_arr
    """
    def __init__(self,num_classes,num_runs):
        #params:
            #int num_classes: the number of classes in this trial
            #int num_runs: the number of runs in this trial
        #Postconditions:
            #initializes CPR_arr np.array shape (NUM_CLASSES, NUM_CLASSES, NUM_RUNS)
            #initializes num_trials_CPR_arr (NUM_CLASSES, NUM_CLASSES)
        self.CPR_arr = np.zeros((num_classes,num_classes,num_runs))
        self.num_trials_CPR_arr = np.zeros((num_classes,num_classes))
    
    def update(self,class_v_query_dict,class_probs,run_number):
        #params:
            #dict class_v_query_dict: dictionary with classes as keys, and number of queries it took to find them
              #as values
            #list class_probs: index corresponds to class, value is the percentage of the class taken from mnist
              #e.g. [12.5, 50, ... , 100] says that 12.5% of the 0s in mnist are used in this run
            #run_number: the run_number
        #Postconditions:
            #The (ixj) layer of CPR_arr corresponding to this run number has been filled with number of queries it
              #took to find each class at its probability
            #num_trials_CPR_arr has been updated to hold the counts of each nonzero element in each pillar of CPR_arr
        
        prob_inds = np.argsort(class_probs)#prob inds looks like [1,4,3,5...,9] where each element is a class,
        #and the index is ordered by probability
        
        for label,num_queries in class_v_query_dict.items():
            # update CPR array itself (not collapsed)
            c = label
            p = np.where(prob_inds==label)[0]
            r = run_number
            self.CPR_arr[c,p,r] = num_queries
        for p, digit in enumerate(prob_inds):
            #update number of entries in each [c,p] cell in self.num_trials_CPR_arr
            self.num_trials_CPR_arr[digit,p] += 1
    
    def calc_av_num_queries(self):
        #returns: array shape (NUM_CLASSES, NUM_CLASSES) i axis is classes, j axis is probabilities
          #each cell, (i,j) contains the average number of queries it took to find class i at probability j
        #Error Catching:
            #in the case that some pillars have no entries, (there is no average) 0 is returned
    #NOTE!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
    #because there are failed trials, some trials will need to decrement the count array, and they will need to be
    #ignored for stats. make a new array at the end? if the trial failed run again with same run number?

        # replace cells where number of trials was 0 with -1's
        temp_arr = np.where(self.num_trials_CPR_arr == 0,-1,self.num_trials_CPR_arr)
        av_num_queries = np.sum(self.CPR_arr,axis = 2)
        av_num_queries = np.divide(av_num_queries,temp_arr)
        # NOTE: any negative answers indicate invalid coverage
        return av_num_queries
    
    def shred_overflow_case(self, class_probs, run_num):
        #params:
            #list<float> class_probs: list of class probs indexed by class for the run to be reset
        #Postconditions:
            #2d array for this run has been reset because of overflow
        self.CPR_arr[:,:,run_num] = 0
        prob_inds = np.argsort(class_probs) #prob inds looks like [1,4,3,5...,9] where each element is a class,
        #and the index is ordered by probability
        for p in range(len(prob_inds)):
            digit = prob_inds[p]
            self.num_trials_CPR_arr[digit, p] -= 1
    
    def calc_av_num_qs_by_representation(self):
        # for calculating average just by level of representation need to do directly from 
        # CPR_arr rather than averaging results of calc_av_num_queries()
        quer_PR = self.CPR_arr.sum(axis=0) #sum all the classes together. Now have probs by runs
        #can now average because we've squashed out all of the holes in CPR
        probs = quer_PR.average(axis=1)
        return probs
    
    def plot_num_qs_vs_class_probs_by_digit(self):
        #Postconditions:
            #Displays plot of num queries (y axis) by class probs (x axis) with one line for each class (color coded)
        
        plt.figure()
        CPR_collapsed = self.calc_av_num_queries() # so 2-D w/ i = classes, j = class prob's
        for c in range(NUM_CLASSES):
            plt.plot(CPR_collapsed[c,:],'s-')
            
        # regenerate class frequencies - each class has half prob of next
        skew_vec = np.empty((NUM_CLASSES,))
        for n in range(NUM_CLASSES):
            skew_vec[n] = 2**n
        skew_vec =skew_vec/(2*np.amax(skew_vec))
        skew_vec = np.round(skew_vec*100,decimals = 1)
        tick_strs = [str(x)+'%' for x in skew_vec]        

        plt.xticks(list(range(NUM_CLASSES)),labels = tick_strs)
#         digit_list = list(range(NUM_CLASSES))
#         digit_list[0] = 'Digit 0'
#         plt.legend(digit_list)
        plt.legend(list(range(NUM_CLASSES)),title = 'Digits')
        plt.xlabel('class probabilities')
        plt.ylabel('number of queries')

In [18]:
class Acc_CPQR:
    """
    The base of Acc_CPQR (class_num, class_prob,query_num,run_num) is a 4D float np.array 
    (acc_CPQR_arr) with:
        classes at 1st index (so i), 
        class_probs at 2nd index (and j), 
        queries at 0th index (and k),
        run_num at 3rd index (and l). 
    For a given run - say l0 - 
       (:,:,:,l0) is a 3d array corresponding to accuracies in a single run.
    There's a column of 0's at (i,j,:,l0) in the array unless class i is at probability j
      for this run- in which case, the column at (i,j,:,l0) contains a vector of accuracies ("pillar") 
      with number of queries as the index of the vector - for this l0 run.
    ATTRIBUTES:
        array acc_CPQR_arr: shape (NUM_CLASSES, NUM_CLASSES, MAX_NUM_QUERIES,NUM_RUNS).
        array num_trials_acc_CPQR_arr: (NUM_CLASSES, NUM_CLASSES) keeps track of the number of 
        nonzero elements in acc_CPQR_arr
    """
    def __init__(self,num_classes,max_num_queries, num_runs):
        #params:
            #int num_classes: the number of classes in this trial
            #int num_classes: the max number of queries allowed in this trial
            #int num_runs: the number of runs in this trial
        #Postconditions:
            #initializes acc_CPQR_arr np.array shape (NUM_CLASSES, NUM_CLASSES,MAX_NUM_QUERIES, NUM_RUNS)
            #initializes num_trials_acc_CPQR_arr (NUM_CLASSES, NUM_CLASSES)
        self.acc_CPQR_arr = np.zeros((num_classes,num_classes,max_num_queries,num_runs))
        self.num_trials_CPQR_arr = np.zeros((num_classes,num_classes))
    
    def update_q(self,class_probs, query_num, run_num, acc_vec=None):
        #params:
            #array acc_vec: shape (NUM_CLASSES,) accuracy[i] corresponds to accuracy of class i.
            #int query_num: number of queries so far
            #list class_probs: index corresponds to class, value is the percentage of the class taken from mnist
              #e.g. [12.5, 50, ... , 100] says that 12.5% of the 0s in mnist are used in this run
            #run_num: the run_number
        #Postconditions:
            #The (ixj) layer of CPR_arr corresponding to this run number has been filled with number of queries it
              #took to find each class at its probability
            #num_trials_CPR_arr has been updated to hold the counts of each nonzero element in each pillar of CPR_arr
            
        prob_inds = np.argsort(class_probs) #prob inds looks like [1,4,3,5...,9] where each element is a class,
        #and the index is ordered by probability
        
        if acc_vec is None:
            self.acc_CPQR_arr[:,:, query_num, run_num] = self.acc_CPQR_arr[:,:,query_num - 1, run_num]
        else:
            for p in range(len(prob_inds)):
                digit = prob_inds[p]
                acc = acc_vec[digit]
                self.acc_CPQR_arr[digit, p, query_num, run_num] = acc
                
    def update_r(self, class_probs):
        #params:
            #list<float> class_probs: list of class probs indexed by class
        #Postconditions:
            #num_trials_CPQR_arr has been updated
        prob_inds = np.argsort(class_probs) #prob inds looks like [1,4,3,5...,9] where each element is a class,
        #and the index is ordered by probability
        for p in range(len(prob_inds)):
            digit = prob_inds[p]
            self.num_trials_CPQR_arr[digit, p] += 1
    
    def shred_overflow_case(self, class_probs, run_num):
        #params:
            #list<float> class_probs: list of class probs indexed by class for the run to be reset
        #Postconditions:
            #3d array for this run has been reset because of overflow
        self.acc_CPQR_arr[:,:,:,run_num] = 0
        prob_inds = np.argsort(class_probs) #prob inds looks like [1,4,3,5...,9] where each element is a class,
        #and the index is ordered by probability
        for p in range(len(prob_inds)):
            digit = prob_inds[p]
            self.num_trials_CPQR_arr[digit, p] -= 1
            
    def reduce_to_PQ(self):
        #returns: array shape (NUM_PROBS, NUM_QUERIES)
        PQR = self.acc_CPQR_arr.sum(axis=0)
        prob_counts = self.num_trials_CPQR_arr.sum(axis=0) # all values the same, this = # runs
        prob_counts = prob_counts[0]
        PQ = PQR.sum(axis=2)/prob_counts
        return PQ

In [19]:
class Method_Stats:
    """
    Class to keep track of statistics generated by a set of runs
    Three levels of stats:
      Method Level: all of the runs combined
      Run Level: data on a partition
      Query Level: data at each query
    ATTRIBUTES:
        Stats attrs:
            CPR cpr: a cpr class to store class v prob v runs
            (it'd be nice if this were named quer_CPR)
            
            array num_queries_when_found: shape (num_runs) at each index, i, holds the number of queries it took
              to find all the classes. If not all classes were found, holds np.nan
              --rename cpr to queries_cpr and make acc_CPQR
              
            array time_per_run: not used in paper stats. Shape (2,num_runs) dtype int at (i, 0) holds the time it took to find 
              all classes for run i, at (i,1) holds the time it took to complete run i
            Acc_CPQR acc_CPQR: accuracy is cell value in 4-D array with indices: query_num,class_num, class_prob,run_num (CPR like - sparse)
              array acc_QR: (max_num_queries, num_runs) holds total accuracy for a run
              NOTE: acc_QR will have duplicates because not all queries result in reclustering (so 
              plateaus in curve)
        Helper attrs:
            int run_number: keeps track of what run_number we are on
    """
    def __init__(self, num_classes, num_runs, max_num_queries):
        #Params:
            #int num_classes: the number of classes for this trial
            #int num_runs: the number of runs for this trial
            #int max_num_queries: the maximum number of queries per run

            self.num_classes = num_classes
            self.num_runs = num_runs
            self.max_num_queries = max_num_queries
            self.overflow_count =0
            #change every run
            self.run_number = np.nan
            self.class_probs = np.nan

            #stats attrs
            self.cpr = CPR(num_classes, num_runs)
            self.acc_CPQR = Acc_CPQR(num_classes,max_num_queries,num_runs)
            self.num_queries_when_found = np.empty((num_runs))
            self.num_queries_when_found[:] = np.nan
            self.time_per_run = np.empty((2, num_runs), dtype='float')
            self.time_per_run[:, :] = np.nan
            self.acc_QR = np.empty((max_num_queries,num_runs)) # acc vs queries v runs
            self.acc_QR[:,:] = np.nan
           
    def update_on_end_of_run(self, class_v_query_dict, time_to_complete_all_queries, time_to_find_all_classes):
        self.acc_CPQR.update_r(self.class_probs)
        self.cpr.update(class_v_query_dict, self.class_probs, self.run_number)
        self.num_queries_when_found[self.run_number] = max(class_v_query_dict.values())
        self.time_per_run[1, self.run_number] = time_to_complete_all_queries
        self.time_per_run[0, self.run_number] = time_to_find_all_classes
    
    def update_on_overflow(self):
        self.acc_CPQR.shred_overflow_case(self.class_probs, self.run_number)
        self.cpr.shred_overflow_case(self.class_probs, self.run_number)
        self.overflow_count += 1
        
    def update_on_query(self, query_num, acc_vec=None,total_acc = None):
        self.acc_CPQR.update_q(self.class_probs, query_num, self.run_number, acc_vec)
        if total_acc:
            self.acc_QR[query_num,self.run_number] = total_acc
        else:
            self.acc_QR[query_num,self.run_number] =   self.acc_QR[query_num-1,self.run_number] 
        
    def to_pickle(self, file_name="method_stats", timestamp=False):
        #params:
            #
        if not os.path.exists("Stat_pickles"):
            os.makedirs("Stat_pickles")
        dt = datetime.now()
        if timestamp:
            file_name += datetime.now().strftime("%m_%d_%H_%M_%S")
        file_name += '.pkl'
        with open("Stat_pickles/"+SPLIT_METHOD+"_"+file_name, 'wb') as file:
            pickle.dump(self, file)
    
    @staticmethod
    def from_pickle(filepath):
        #params: str filepath: the path to a pkl file of a Method_Stats object
        #returns: the unpickled Method_Stats object
        with open(filepath, 'rb') as file:
            return pickle.load(file)
    
    @staticmethod
    def merge(stats1, stats2):
        #params:
            #Method_Stats stats1: a method stats object to merge
            #Method_Stats stats2: a method stats object to merge
        #Preconditions:
            #Both classes must have the same exact same classes (and by extension the same num_classes)
            #Both classes must have same max_num_queries
        #returns:
            #Method_Stats an object that contains the data from both inputs. Data from stats1 is placed before that
            #of stats2 in the any array
            #run_number is initialized to 0


        merged_stats = Method_Stats(stats1.num_classes, stats1.num_runs + stats2.num_runs, 
                                    stats1.max_num_queries)

        merged_stats.cpr.CPR_arr = np.concatenate((stats1.cpr.CPR_arr, stats2.cpr.CPR_arr), axis=2)
        merged_stats.cpr.num_trials_CPR_arr = stats1.cpr.num_trials_CPR_arr + stats2.cpr.num_trials_CPR_arr
        merged_stats.acc_CPQR.acc_CPQR_arr = np.concatenate((stats1.acc_CPQR.acc_CPQR_arr, stats2.acc_CPQR.acc_CPQR_arr),axis=3)
        merged_stats.acc_CPQR.num_trials_CPQR_arr = stats1.acc_CPQR.num_trials_CPQR_arr + stats2.acc_CPQR.num_trials_CPQR_arr

        merged_stats.num_queries_when_found = np.concatenate((stats1.num_queries_when_found, stats2.num_queries_when_found))
        merged_stats.time_per_run = np.concatenate((stats1.time_per_run, stats2.time_per_run), axis=1)
        merged_stats.acc_QR = np.concatenate((stats1.acc_QR, stats2.acc_QR), axis=1)
        merged_stats.overflow_count = stats1.overflow_count+stats2.overflow_count
        return merged_stats


In [ ]:
# this is a "trial" - all the runs
for trial in range(NUM_TRIALS):
    method_stats = Method_Stats(NUM_CLASSES, NUM_RUNS, MAX_NUM_QUERIES)
    run = 0
    while run < NUM_RUNS:
        print("run #: ",run)
        skewed_data,skewed_labels,skew_vec,class_probs = skew_data(raw_data, raw_labels)
        # need a global - and data is already
        data = skewed_data[::DECIMATION_FACTOR,:]
        labels = skewed_labels[::DECIMATION_FACTOR]    #initialize a new partition
        #update run level attribute of method_stats
        method_stats.class_probs = class_probs
        method_stats.run_number = run
        data_partition = Partition(data,labels,SPLIT_METHOD, method_stats)
        data_partition.run(NUM_CLASSES,MAX_NUM_QUERIES)
        print("accuracy of: ", method_stats.acc_QR[-1, run])
        print("all classes found at:", method_stats.time_per_run[0,run])
        print("Total time for ", MAX_NUM_QUERIES," queries:", method_stats.time_per_run[1, run])
        if len(data_partition.class_v_query_dict) < NUM_CLASSES:
            method_stats.update_on_overflow()
        else:
            run+=1
    print("query overflow count: ", method_stats.overflow_count)
    method_stats.to_pickle(timestamp=True)

run #:  0
10
20
30
40
50
60
70
80
90
100
110
120
130
140
150
160
170
180
190
200
210
220
230
240
250
260
270
{6: 1, 4: 2, 8: 4, 5: 5, 3: 6, 2: 7, 0: 30, 9: 74, 7: 121, 1: 273}
all 10 classes found at 273 queries
accuracy of:  nan
all classes found at: 89.18820834159851
Total time for  1000  queries: 89.18870878219604
query overflow count:  0
run #:  0
10
20
30
40
50
60
70
80
90
100
110
120
130
140
{4: 1, 3: 2, 6: 4, 0: 5, 8: 9, 5: 25, 9: 28, 7: 29, 1: 76, 2: 150}
all 10 classes found at 150 queries
150
accuracy of:  nan
all classes found at: 105.06179404258728
Total time for  1000  queries: 105.06198072433472
query overflow count:  0
run #:  0
10
20
30
40
50
60
70
80
{7: 1, 2: 2, 5: 3, 3: 6, 9: 14, 0: 15, 8: 21, 6: 43, 4: 49, 1: 88}
all 10 classes found at 88 queries
accuracy of:  nan
all classes found at: 69.6998963356018
Total time for  1000  queries: 69.70002460479736
query overflow count:  0
run #:  0
10
20
30
40
50
60
70
80
90
100
110
120
130
140
150
160
170
180
190
200
210
220
23

420
430
440
450
460
470
480
490
500
510
520
530
540
550
560
570
580
590
600
610
620
630
640
650
660
670
680
690
700
710
720
{2: 1, 5: 2, 8: 4, 0: 6, 4: 9, 7: 23, 1: 32, 9: 57, 6: 161, 3: 724}
all 10 classes found at 724 queries
accuracy of:  nan
all classes found at: 113.09907460212708
Total time for  1000  queries: 113.09921765327454
query overflow count:  0
run #:  0
10
20
30
40
50
60
70
80
90
100
{0: 1, 6: 2, 2: 5, 9: 7, 8: 9, 7: 14, 5: 23, 1: 54, 3: 66, 4: 106}
all 10 classes found at 106 queries
accuracy of:  nan
all classes found at: 64.7032561302185
Total time for  1000  queries: 64.70404410362244
query overflow count:  0
run #:  0
10
20
30
40
50
60
70
80
90
100
{2: 1, 5: 2, 3: 3, 0: 8, 9: 16, 6: 19, 4: 48, 8: 65, 7: 96, 1: 106}
all 10 classes found at 106 queries
accuracy of:  nan
all classes found at: 50.34413123130798
Total time for  1000  queries: 50.344281911849976
query overflow count:  0
run #:  0
10
20
30
40
50
60
70
80
90
100
110
120
130
{0: 1, 4: 2, 3: 10, 1: 12, 5: 13